# The Third Agent

## Fine-tuned LLM using QLoRA with Llama 3.1 as the base model

First, here's a link to Google Colab set up for training with QLoRA

https://colab.research.google.com/drive/1IqxWtUzuV5ks2kS1oO4Mge3Mf1o3rhRj

And here's a link to Google Colab set up for inference:

https://colab.research.google.com/drive/1shI0i5QiMWL8fSmM-VcBI7RT5NjzZJ17

Once this is set up, I have this running on Modal

If you want to do this too, head over to modal.com to set up your free starter account with free credit

In [1]:
import pickle
from evaluator import evaluate
import modal

In [2]:
with open('../test.pkl', 'rb') as file:
    test = pickle.load(file)

During the class I might visit this URL to show the code deployed on Modal:

https://modal.com/apps/ed-donner/main/ap-stiZMq9syc9zikKRoLnRor?functionId=fu-LumBocLb9rvkzuIUBQGn42&activeTab=functions

In [3]:
# For you to experiment after the class: below we set up and deploy our proprietary LLM over modal
# Here we execute it directly

Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

# Keep awake for 20 mins
pricer.update_autoscaler(scaledown_window=1200)

reply = pricer.price.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(reply)

133.0


In [4]:
# Generations of iPad pro

print("iPad Pro 1st gen estimate:", pricer.price.remote("iPad pro 1st generation"))
print("iPad Pro 6th gen estimate:", pricer.price.remote("iPad pro 6th generation"))

iPad Pro 1st gen estimate: 299.0
iPad Pro 6th gen estimate: 799.0


In [5]:
def fine_tuned_pricer(item):
    description = item.test_prompt().replace("How much does this cost to the nearest dollar?\n\n","").replace("\n\nPrice is $","")
    price = pricer.price.remote(description)
    return price

In [6]:
evaluate(fine_tuned_pricer, test, 50)

  0%|          | 0/50 [00:00<?, ?it/s]

$38 $43 $31 $250 $2 $13 $31 $10 $314 $12 $65 $1 $30 $32 $100 $2 $40 $29 $120 $0 $61 $15 $6 $60 $2 $1 $13 $48 $5 $54 $46 $22 $3 $90 $270 $84 $0 $120 $7 $25 $0 $8 $2 $301 $25 $46 $0 $9 $22 $90 

In [7]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [8]:
from price_agents.specialist_agent import SpecialistAgent

agent = SpecialistAgent()
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $133.00


133.0

In [9]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $299.00


299.0

# For you to get this to work yourself

## We need to set your HuggingFace Token as a secret in Modal

1. Go to modal.com, sign in and go to your dashboard
2. Click on Secrets in the nav bar
3. Create new secret, click on Hugging Face
4. Fill in your HF_TOKEN where it prompts you


In [ ]:
# First time: uncomment and run the line below
# !modal setup

# Deploying and running:

From a command line, `modal deploy xxx` will deploy your code as a Deployed App

This is how you could package your AI service behind an API to be used in a Production System.

You can also build REST endpoints easily, although we won't cover that as we'll be calling direct from Python.

In [ ]:
!uv run modal deploy pricer_service2

In [ ]:
Pricer = modal.Cls.lookup("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(reply)